# Fine-Tuning YOLO11 pour l'Inspection Scolaire

Detecte 6 equipements : Tables, Chaises, Ordinateurs, Imprimantes, Videoprojecteurs, Extincteurs.

---
## Avant de commencer
1. **Runtime -> Change runtime type -> T4 GPU**
2. Executez les cellules dans l'ordre
3. Modele final sauvegarde dans Google Drive

In [ ]:
# Installation des dependances
!pip install -q ultralytics
from ultralytics import YOLO
from pathlib import Path
import os, sys, yaml, shutil
print('OK')

In [ ]:
# Monter Google Drive
from google.colab import drive
drive.mount('/content/drive')
DRIVE_PATH = '/content/drive/MyDrive/inspection_scolaire_yolo'
os.makedirs(DRIVE_PATH, exist_ok=True)

---
## Etape 1 : Upload du dataset

In [ ]:
# Upload dataset.zip
from google.colab import files
print('Uploader dataset.zip (data.yaml + train/ + val/ + test/)')
uploaded = files.upload()
for fn in uploaded.keys():
    !unzip -o {fn} -d /content/dataset/

yaml_path = list(Path('/content/dataset').rglob('data.yaml'))
if yaml_path:
    yaml_path = yaml_path[0]
else:
    yaml_path = list(Path('/content').rglob('data.yaml'))[0]
    !cp -r {yaml_path.parent} /content/dataset/
    yaml_path = Path('/content/dataset/data.yaml')
print(f'data.yaml: {yaml_path}')

In [ ]:
# Exploration du dataset
with open(yaml_path) as f:
    data_cfg = yaml.safe_load(f)
print(f'Classes: {data_cfg["nc"]} - {data_cfg["names"]}')
for s in ['train','val','test']:
    d = yaml_path.parent / data_cfg.get(s, f'{s}/images')
    if d.exists(): print(f'{s}: {len(list(d.glob("*")))} images')

---
## Etape 2 : Entrainement

In [ ]:
# Parametres
EPOCHS, BATCH_SIZE, IMG_SIZE = 150, 32, 640
BASE_MODEL, PATIENCE, LR = 'yolo11n.pt', 30, 0.001
print('GPU:', !nvidia-smi --query-gpu=name --format=csv,noheader)

In [ ]:
# Lancer l'entrainement
print('Entrainement YOLO...')
model = YOLO(BASE_MODEL)
model.train(data=str(yaml_path), epochs=EPOCHS, batch=BATCH_SIZE,
            imgsz=IMG_SIZE, patience=PATIENCE, lr0=LR,
            device='cuda', project='/content/runs',
            name='inspection_scolaire', exist_ok=True,
            pretrained=True, augment=True, verbose=True)
print('Entrainement termine !')

---
## Etape 3 : Evaluation

In [ ]:
# Courbes d'entrainement
from IPython.display import Image as IPImage
rp = Path('/content/runs/inspection_scolaire')
for img in ['results.png','confusion_matrix.png','F1_curve.png','PR_curve.png']:
    f = rp / img
    if f.exists(): display(IPImage(filename=str(f), width=800))

In [ ]:
# Test sur une image
from google.colab.patches import cv2_imshow
model = YOLO('/content/runs/inspection_scolaire/weights/best.pt')
test_imgs = list(Path('/content/dataset/test/images').glob('*'))
if test_imgs:
    results = model(str(test_imgs[0]))
    cv2_imshow(results[0].plot())
    for box in results[0].boxes:
        print(f'{results[0].names[int(box.cls[0])]}: {float(box.conf[0]):.2%}')

---
## Etape 4 : Sauvegarde

In [ ]:
# Sauvegarder dans Google Drive
shutil.copytree('/content/runs/inspection_scolaire',
                f'{DRIVE_PATH}/runs_inspection_scolaire',
                dirs_exist_ok=True)
shutil.copy2('/content/runs/inspection_scolaire/weights/best.pt',
             f'{DRIVE_PATH}/best.pt')
print(f'Modele: {DRIVE_PATH}/best.pt')

In [ ]:
# Telecharger best.pt
from google.colab import files
files.download('/content/runs/inspection_scolaire/weights/best.pt')

---
## Integration

Dans `yolo_service.py` :
```python
class YoloService:
    def __init__(self):
        self.model = YOLO('best.pt')
    def detect(self, image_path):
        results = self.model(image_path)
        return [{
            'class_id': int(b.cls[0]),
            'class_name': results[0].names[int(b.cls[0])],
            'confidence': float(b.conf[0]),
            'bbox': b.xyxy[0].tolist(),
        } for b in results[0].boxes]
```

---
## Resume
1. Upload dataset -> OK
2. Entrainement YOLO -> OK
3. Evaluation -> OK
4. Sauvegarde Drive -> OK